# Compare average acceleration over hours of the day between matched controls and cases

In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import pylab as plt

from scipy import stats
from statannot import add_stat_annotation
import pingouin as pg

from importlib import reload
import sys
import os
sys.path.insert(1,'../')
import plots

In [ ]:
reload(plots)
image_path = plots.set_dir()
colors = plots.set_colors()
data_path = '/scratch/c.c21013066/data/ukbiobank/phenotypes/accelerometer/'

In [ ]:
dfs_new = pd.read_csv(f'{data_path}matched_all_HCnoOsteo.csv',index_col=0)
for name in ['Osteoarthritis','ParkinsonDisease','Depression','AllCauseDementia']:
    if name == 'ParkinsonDisease':
        matched_sample = dfs_new[dfs_new['diagnosis']==name]

        time_matched = matched_sample[np.hstack([name,'diagnosis','diagnosis_prod','diagnosis_prod_conservative','Group',matched_sample.filter(regex='Average').columns,'visit_age',f'{name}_age','time_to_diagnosis','accelerometry_age','acc_time_since_diagnosis'])]
        time_matched = time_matched.melt(id_vars=[name,'accelerometry_age','time_to_diagnosis','acc_time_since_diagnosis','diagnosis','diagnosis_prod','diagnosis_prod_conservative',f'{name}_age','Group'],
                                         value_vars=matched_sample.filter(regex='Average').columns)

        matched = 'matched'
        for diag in ['Group']:
            time_matched['match'] = time_matched[diag].replace(['Prodromal','Diseased','Healthy_Prodromal',
                                                                'Healthy_Diseased'],
                                                          ['prodromal','diseased','prodromal','diseased'])
            time_matched['status'] = time_matched[diag].replace(['Prodromal','Diseased','Healthy_Prodromal',
                                                                 'Healthy_Diseased'],
                                                          ['prodromal','diseased','healthy','healthy'])
            xnames = matched_sample.filter(regex='Average').columns.str.replace('Average','').str.replace('___','-')
            labels = time_matched[diag].value_counts()//len(xnames)
            
            pvals = pd.DataFrame(columns=['t','pvalue'],
                                 index=pd.MultiIndex.from_product([['diagHC','prodHC'],
                                                                   np.unique(time_matched['variable'])],
                                                                  names=['group','hour']))
            statistics = pd.DataFrame(columns=['T','dof','alternative','p-value','CI95%','cohen-d','BF10','power'],
                                               index=pd.MultiIndex.from_product([['diagHC','prodHC'],
                                                                   np.unique(time_matched['variable'])],
                                                                  names=['group','hour']))
            for hour in np.unique(time_matched['variable']):
                dat = time_matched[time_matched['variable']==hour]
                stat = pg.ttest(dat.loc[dat['Group']=='Healthy_Diseased','value'],dat.loc[dat['Group']=='Diseased',
                                                                                          'value'])
                statistics.loc[('diagHC',hour),:] = stat.values
                t,p = stats.ttest_ind(dat.loc[dat['Group']=='Healthy_Diseased','value'],dat.loc[dat['Group']=='Diseased',
                                                                                                'value'])
                pvals.loc[('diagHC',hour),'t'] = t
                pvals.loc[('diagHC',hour),'pvalue'] = p
                stat = pg.ttest(dat.loc[dat['Group']=='Healthy_Prodromal','value'],dat.loc[dat['Group']=='Prodromal',
                                                                                           'value'])
                statistics.loc[('prodHC',hour),:] = stat.values
                t,p = stats.ttest_ind(dat.loc[dat['Group']=='Healthy_Prodromal','value'],dat.loc[dat['Group']=='Prodromal',
                                                                                                 'value'])
                pvals.loc[('prodHC',hour),'t'] = t
                pvals.loc[('prodHC',hour),'pvalue'] = p
            # save for supplemental Table
            pvals.to_csv(f'{data_path}rawdaytime_Parkinson_pvals.csv')
            statistics.to_csv(f'{data_path}rawdaytime_Parkinson_statistics_HCnoOsteo.csv')
            print(pvals.to_latex())
            print(0.05/pvals.shape[0]/2)
            fig = plt.figure(figsize=(12,5))
            plots.plot_context()
            ax = sns.lineplot(x='variable',y='value',data=time_matched,hue='status',palette=colors.palette[:3],
                              hue_order=['healthy','prodromal','diseased'],
                             style='match',style_order=['diseased','prodromal'])
            prod = labels["Prodromal"]
            ax.legend(handles=ax.lines[:4][::-1], labels=[f"Matched Controls for Diagnosed ({labels['Healthy_Diseased']})",
                                                f"Matched Controls for Prodromal ({labels['Healthy_Prodromal']})",
                                                f"Prodromal ({prod})",
                                                f"Diagnosed ({labels['Diseased']})"
                                                    ][::-1],bbox_to_anchor=[1,1])
            ax.set_xticklabels(xnames,rotation=90);
            ax.set_xlabel('time [1h intervals]')
            ax.set_ylabel('average acceleration [milligal]');
            plt.subplots_adjust(bottom=0.1)
            #plt.savefig(f'{image_path}/average_acc_day_{matched}_{diag}_{name}_HCnoOsteo.pdf',bbox_inches='tight',dpi=300)
            #plt.savefig(f'{image_path}/average_acc_day_{matched}_{diag}_{name}_HCnoOsteo.png',bbox_inches='tight',dpi=300)
            plt.show()

In [ ]:
statistics

In [ ]:
# age difference
pg.ttest(matched_sample.loc[matched_sample['Group']=='Prodromal','accelerometry_age'],
         matched_sample.loc[matched_sample['Group']=='Diseased','accelerometry_age'])